# O6 — Quantization sensitivity bound (O5 measurement)

Quantization perturbs logits. O5's `gold_first_token_rank` is a rank over the full vocabulary, so **4-bit ranks may not be comparable to fp16**. This notebook bounds the error on Qwen2.5-1.5B-Instruct.

**Design**
1. Draw a **fixed-seed stratified subsample of 60** Probe-1 cells (20 GSM + 20 ALGO + 20 BW) from the same O5 item universe.
2. Run the **identical O5 teacher-forced gold-sequence measurement** at three precisions on `Qwen/Qwen2.5-1.5B-Instruct`:
   - **fp16** (reference)
   - **8-bit** (bitsandbytes)
   - **4-bit NF4** (`compute_dtype=float16`)
3. For each precision pair, report: mean |Δ mean_logprob|, Spearman(mean_logprob), median |Δ gold_first_token_rank|, 95th percentile |Δ rank|.

**T4:** fp16 only for unquantized loads; `attn_implementation="sdpa"`; no FlashAttention-2; no bf16.

**Decision rule (methods):** if the **median absolute rank shift** for fp16↔4-bit exceeds roughly **50** positions, drop all 4-bit rank measurements from the paper and keep only fp16 (state this explicitly).

**Outputs**
- `colab_out/O6_quantization_sensitivity.csv` — pairwise summary (required)
- `colab_out/O6_quantization_sensitivity_items.csv` — every per-item score (audit; do not lose intermediates)
- `colab_out/O6_quantization_sensitivity_summary.txt` — one-paragraph usability verdict

`LIMIT` (setup cell) shrinks the per-family draw for smoke tests (e.g. `LIMIT=2` → 6 items).


In [ ]:
# Colab T4: bitsandbytes for quantized loads. Restart the runtime if
# bitsandbytes was just installed and the kernel has not picked it up.
import sys
import subprocess
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.44",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "pandas",
        "scipy",
        "networkx",
        "tqdm",
        "huggingface_hub",
    ]
)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ── knobs ────────────────────────────────────────────────────────────────
# Set LIMIT to an int for a smoke test (e.g. 2 items per family). None = full run.
LIMIT = None
DRY_RUN = False          # True: skip GPU, write placeholder rows (pipeline check)
RESUME = True

# Private GitHub clone (Colab secret GITHUB_TOKEN, or env). Public clone works
# without a token. If this notebook is already inside the repo, clone is skipped.
REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
GH_TOKEN = _secret("GITHUB_TOKEN")

# Llama-3.1-8B-Instruct is gated: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as _hf_exc:
        print("[setup] huggingface login skipped:", _hf_exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "probes" / "contamination" / "verify.py").is_file() and (
        p / "data" / "problems" / "question_bank_gsm.csv"
    ).is_file()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    colab = Path("/content/retrieval-vs-computation")
    if _looks_like_repo(colab):
        return colab
    return colab

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    cmd = ["git", "clone", "--depth", "1", url, str(REPO_ROOT)]
    subprocess.check_call(cmd)
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), (
    f"Could not find probes/ + question banks under {REPO_ROOT}. "
    "Clone the retrieval-vs-computation repo, or set RVC_REPO_URL / GITHUB_TOKEN."
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = Path("/content/colab_out") if Path("/content").exists() else (REPO_ROOT / "colab_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] OUT_DIR={OUT_DIR}")
print(f"[setup] LIMIT={LIMIT} DRY_RUN={DRY_RUN} RESUME={RESUME}")


## Stratified 60-item subsample + O5 measurement primitives

Same Appendix-N prompt and teacher-forced gold continuation as O5. Subsample seed is fixed (`SUBSAMPLE_SEED=42`).


In [ ]:
from __future__ import annotations

import csv
import gc
import json
import random
import re
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PRECISIONS = ("fp16", "int8", "nf4")
SUBSAMPLE_SEED = 42
N_PER_FAMILY = 20  # 20 × 3 = 60; overridden downward when LIMIT is set
RANK_DROP_THRESHOLD = 50  # median |Δ rank| fp16↔nf4; methods kill criterion

PROBE1_TEMPLATE = (
    "Solve the following problem exactly and provide only the final answer "
    "in the required output format. Problem: {problem}. Format instruction: "
    "{family_specific_output_format}."
)
FAMILY_FORMAT = {
    "GSM": (
        "Write the final numerical answer on its own line as #### <number>. "
        "No other text after that tag."
    ),
    "ALGO": (
        "Follow the problem's required output format exactly "
        "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
    ),
    "BW": (
        "A numbered list of actions only. Each action must be one of the "
        "permitted operators with their arguments. No explanation."
    ),
}
VARIANTS = ("canonical", "W1", "W2", "W3", "W4", "W5", "W6")

O6_ITEMS_CSV = OUT_DIR / "O6_quantization_sensitivity_items.csv"
O6_CSV = OUT_DIR / "O6_quantization_sensitivity.csv"
O6_SUMMARY_TXT = OUT_DIR / "O6_quantization_sensitivity_summary.txt"

ITEM_COLUMNS = [
    "family", "problem_id", "variant", "precision", "model",
    "n_gold_tokens", "sum_logprob", "mean_logprob",
    "gold_first_token_rank", "gold_first_token_logprob", "prompt_n_tokens",
]
PAIR_COLUMNS = [
    "precision_a", "precision_b", "n_items",
    "mean_abs_diff_mean_logprob", "spearman_mean_logprob", "spearman_pvalue",
    "median_abs_rank_shift", "p95_abs_rank_shift",
    "max_abs_rank_shift",
]


def _norm_vt(v: str) -> str:
    v = str(v).strip()
    return "canonical" if v.lower() == "canonical" else v.upper()


def _strip_csv_quotes(text: str) -> str:
    s = str(text)
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]
    return s


def build_prompt(problem_text: str, family: str) -> str:
    return PROBE1_TEMPLATE.format(
        problem=problem_text.strip(),
        family_specific_output_format=FAMILY_FORMAT[family],
    )


def _load_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str).fillna("")
    df["problem_id"] = df["problem_id"].astype(str).str.strip()
    df["variant_type"] = df["variant_type"].map(_norm_vt)
    df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
    df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
    return df


def load_o5_universe() -> list[dict[str, Any]]:
    """Full O5 cell universe (family × problem × variant present in banks)."""
    specs = [
        ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
        ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
        ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
    ]
    items: list[dict[str, Any]] = []
    for family, path in specs:
        df = _load_bank(path)
        can_ids = set(df.loc[df.variant_type == "canonical", "problem_id"])
        for _, row in df.iterrows():
            pid = str(row["problem_id"])
            vt = str(row["variant_type"])
            if vt not in VARIANTS or pid not in can_ids:
                continue
            items.append(
                {
                    "family": family,
                    "problem_id": pid,
                    "variant": vt,
                    "problem_text": str(row["problem_text"]),
                    "gold": str(row["correct_answer"]),
                }
            )
    return items


def stratified_subsample(
    universe: list[dict[str, Any]],
    n_per_family: int,
    seed: int,
) -> list[dict[str, Any]]:
    rng = random.Random(seed)
    out: list[dict[str, Any]] = []
    for fam in ("GSM", "ALGO", "BW"):
        pool = [x for x in universe if x["family"] == fam]
        if not pool:
            raise RuntimeError(f"empty pool for {fam}")
        k = min(n_per_family, len(pool))
        out.extend(rng.sample(pool, k=k))
    out.sort(key=lambda x: (x["family"], x["problem_id"], x["variant"]))
    return out


_n_per = N_PER_FAMILY if LIMIT is None else min(int(LIMIT), N_PER_FAMILY)
UNIVERSE = load_o5_universe()
ITEMS = stratified_subsample(UNIVERSE, _n_per, SUBSAMPLE_SEED)
print(
    f"[sample] n={len(ITEMS)}  seed={SUBSAMPLE_SEED}  "
    f"n_per_family={_n_per}  universe={len(UNIVERSE)}"
)
print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())
(OUT_DIR / "O6_subsample_manifest.json").write_text(
    json.dumps(
        {
            "model": MODEL_ID,
            "seed": SUBSAMPLE_SEED,
            "n_per_family": _n_per,
            "n_items": len(ITEMS),
            "items": [
                {"family": x["family"], "problem_id": x["problem_id"], "variant": x["variant"]}
                for x in ITEMS
            ],
        },
        indent=2,
    )
)
print(f"[sample] wrote {OUT_DIR / 'O6_subsample_manifest.json'}")


## Teacher-forced metrics (identical to O5) + precision loaders


In [ ]:
def wrap_chat(tokenizer, user_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
        tokenize=False,
    )


def resolve_continuation(
    tokenizer,
    prompt: str,
    answer: str,
) -> tuple[list[int], list[int], str]:
    def enc(text: str) -> list[int]:
        return tokenizer.encode(text, add_special_tokens=False)

    prompt_ids = enc(prompt)
    answer = str(answer)
    if not answer:
        return prompt_ids, [], "EMPTY"
    candidates: list[tuple[str, list[int], int]] = []
    for sep in ("", " "):
        joint = enc(prompt + sep + answer)
        if len(joint) <= len(prompt_ids):
            continue
        if joint[: len(prompt_ids)] != prompt_ids:
            continue
        rest = joint[len(prompt_ids) :]
        candidates.append((sep, rest, len(joint)))
    if not candidates:
        return prompt_ids, enc(answer), "FALLBACK"
    candidates.sort(key=lambda c: c[2])
    sep, rest, _ = candidates[0]
    return prompt_ids, rest, repr(sep)


@torch.inference_mode()
def teacher_forced_metrics(
    model,
    tokenizer,
    device,
    user_text: str,
    gold_text: str,
) -> dict[str, Any]:
    prompt = wrap_chat(tokenizer, user_text)
    prompt_ids, gold_ids, sep_note = resolve_continuation(tokenizer, prompt, gold_text)
    n_prompt = len(prompt_ids)
    n_gold = len(gold_ids)
    if n_gold == 0:
        return {
            "n_gold_tokens": 0,
            "sum_logprob": float("nan"),
            "mean_logprob": float("nan"),
            "gold_first_token_rank": -1,
            "gold_first_token_logprob": float("nan"),
            "prompt_n_tokens": n_prompt,
            "sep_note": sep_note,
        }
    if DRY_RUN or model is None:
        # Deterministic fake ranks keyed by precision label length — only for pipeline checks.
        return {
            "n_gold_tokens": n_gold,
            "sum_logprob": -0.1 * n_gold,
            "mean_logprob": -0.1,
            "gold_first_token_rank": 1,
            "gold_first_token_logprob": -0.1,
            "prompt_n_tokens": n_prompt,
            "sep_note": "DRY_RUN",
        }

    input_ids = torch.tensor([prompt_ids + gold_ids], dtype=torch.long, device=device)
    out = model(input_ids=input_ids, use_cache=False)
    logits = out.logits[0]
    gold_logits = logits[n_prompt - 1 : n_prompt + n_gold - 1]
    log_probs = F.log_softmax(gold_logits.float(), dim=-1)
    gold_t = torch.tensor(gold_ids, device=device, dtype=torch.long)
    tok_lp = log_probs.gather(1, gold_t.unsqueeze(1)).squeeze(1)
    sum_lp = float(tok_lp.sum().item())
    mean_lp = sum_lp / n_gold
    first_logits = gold_logits[0].float()
    first_tid = int(gold_ids[0])
    first_lp = float(F.log_softmax(first_logits, dim=-1)[first_tid].item())
    rank = int((first_logits > first_logits[first_tid]).sum().item()) + 1
    del out, logits, input_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "n_gold_tokens": n_gold,
        "sum_logprob": round(sum_lp, 6),
        "mean_logprob": round(mean_lp, 6),
        "gold_first_token_rank": rank,
        "gold_first_token_logprob": round(first_lp, 6),
        "prompt_n_tokens": n_prompt,
        "sep_note": sep_note,
    }


def load_model(precision: str):
    assert torch.cuda.is_available() or DRY_RUN, "GPU required (Colab T4) unless DRY_RUN."
    tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN or True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    if DRY_RUN:
        print(f"[model] DRY_RUN skip load: {MODEL_ID} ({precision})")
        return tok, None, torch.device("cpu")

    common = dict(
        device_map="auto",
        token=HF_TOKEN or True,
        attn_implementation="sdpa",
    )
    if precision == "fp16":
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            **common,
        )
        label = "fp16 unquantized + sdpa"
    elif precision == "int8":
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb,
            torch_dtype=torch.float16,
            **common,
        )
        label = "int8 bitsandbytes + sdpa"
    elif precision == "nf4":
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb,
            torch_dtype=torch.float16,
            **common,
        )
        label = "nf4 4-bit (compute_dtype=float16) + sdpa"
    else:
        raise ValueError(precision)
    mdl.eval()
    device = next(mdl.parameters()).device
    print(f"[model] {MODEL_ID}  {label}  device={device}")
    return tok, mdl, device


def unload(mdl):
    if mdl is None:
        return
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def score_primary(model, tokenizer, device, item: dict, precision: str) -> dict[str, Any]:
    user = build_prompt(item["problem_text"], item["family"])
    m = teacher_forced_metrics(model, tokenizer, device, user, item["gold"])
    # DRY_RUN: inject precision-dependent rank noise so pairwise shifts are nonzero in pipeline checks
    if DRY_RUN or model is None:
        bump = {"fp16": 0, "int8": 3, "nf4": 80}[precision]
        m = dict(m)
        m["gold_first_token_rank"] = 1 + bump
        m["mean_logprob"] = round(-0.1 - 0.01 * bump, 6)
        m["sum_logprob"] = round(m["mean_logprob"] * m["n_gold_tokens"], 6)
    return {
        "family": item["family"],
        "problem_id": item["problem_id"],
        "variant": item["variant"],
        "precision": precision,
        "model": MODEL_ID,
        "n_gold_tokens": m["n_gold_tokens"],
        "sum_logprob": m["sum_logprob"],
        "mean_logprob": m["mean_logprob"],
        "gold_first_token_rank": m["gold_first_token_rank"],
        "gold_first_token_logprob": m["gold_first_token_logprob"],
        "prompt_n_tokens": m["prompt_n_tokens"],
    }


## Run all three precisions → pairwise sensitivity + verdict

Decision: if median |Δ rank| for **fp16 vs nf4** > ~50, 4-bit ranks are **not usable** in the paper.


In [ ]:
def _done_keys(path: Path) -> set[tuple[str, str, str, str]]:
    if not path.exists():
        return set()
    df = pd.read_csv(path, dtype=str)
    need = {"precision", "family", "problem_id", "variant"}
    if not need.issubset(df.columns):
        return set()
    return {
        (str(r.precision), str(r.family), str(r.problem_id), str(r.variant))
        for r in df.itertuples(index=False)
    }


def append_item_rows(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    write_header = not path.exists()
    with path.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=ITEM_COLUMNS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in ITEM_COLUMNS})


done = _done_keys(O6_ITEMS_CSV) if RESUME else set()
print(f"[resume] {len(done)} item-rows in {O6_ITEMS_CSV}")

for precision in PRECISIONS:
    pending = [
        it
        for it in ITEMS
        if (precision, it["family"], it["problem_id"], it["variant"]) not in done
    ]
    print(f"\n=== {MODEL_ID} @ {precision}  pending={len(pending)}/{len(ITEMS)} ===")
    if not pending:
        continue
    tok, mdl, device = load_model(precision)
    buf: list[dict[str, Any]] = []
    try:
        for it in tqdm(pending, desc=f"{precision}"):
            buf.append(score_primary(mdl, tok, device, it, precision))
            if len(buf) >= 20:
                append_item_rows(O6_ITEMS_CSV, buf)
                buf.clear()
        append_item_rows(O6_ITEMS_CSV, buf)
    finally:
        unload(mdl)

items_df = pd.read_csv(O6_ITEMS_CSV)
print(f"[items] n={len(items_df)}  precisions={sorted(items_df.precision.unique())}")


def pairwise_row(df: pd.DataFrame, a: str, b: str) -> dict[str, Any]:
    wide = (
        df[df.precision.isin([a, b])]
        .pivot_table(
            index=["family", "problem_id", "variant"],
            columns="precision",
            values=["mean_logprob", "gold_first_token_rank"],
            aggfunc="first",
        )
        .dropna()
    )
    # Flatten MultiIndex columns
    lp_a = wide[("mean_logprob", a)].astype(float)
    lp_b = wide[("mean_logprob", b)].astype(float)
    rk_a = wide[("gold_first_token_rank", a)].astype(float)
    rk_b = wide[("gold_first_token_rank", b)].astype(float)
    abs_lp = (lp_a - lp_b).abs()
    abs_rk = (rk_a - rk_b).abs()
    if len(lp_a) >= 2 and lp_a.nunique() > 1 and lp_b.nunique() > 1:
        spearman_r, spearman_p = stats.spearmanr(lp_a, lp_b)
    elif len(lp_a) >= 2:
        spearman_r, spearman_p = float("nan"), float("nan")
    else:
        spearman_r, spearman_p = float("nan"), float("nan")
    return {
        "precision_a": a,
        "precision_b": b,
        "n_items": int(len(wide)),
        "mean_abs_diff_mean_logprob": round(float(abs_lp.mean()), 6),
        "spearman_mean_logprob": (
            round(float(spearman_r), 6) if spearman_r == spearman_r else ""
        ),
        "spearman_pvalue": (
            round(float(spearman_p), 6) if spearman_p == spearman_p else ""
        ),
        "median_abs_rank_shift": round(float(abs_rk.median()), 3),
        "p95_abs_rank_shift": round(float(np.percentile(abs_rk, 95)), 3),
        "max_abs_rank_shift": round(float(abs_rk.max()), 3),
    }


pairs = [("fp16", "int8"), ("fp16", "nf4"), ("int8", "nf4")]
pair_rows = [pairwise_row(items_df, a, b) for a, b in pairs]
pair_df = pd.DataFrame(pair_rows, columns=PAIR_COLUMNS)
pair_df.to_csv(O6_CSV, index=False)
print("\n=== O6_quantization_sensitivity.csv ===")
print(pair_df.to_string(index=False))

fp16_nf4 = next(r for r in pair_rows if r["precision_a"] == "fp16" and r["precision_b"] == "nf4")
med = float(fp16_nf4["median_abs_rank_shift"])
p95 = float(fp16_nf4["p95_abs_rank_shift"])
mad_lp = float(fp16_nf4["mean_abs_diff_mean_logprob"])
sp = fp16_nf4["spearman_mean_logprob"]
n = int(fp16_nf4["n_items"])
drop_4bit_ranks = med > RANK_DROP_THRESHOLD

if drop_4bit_ranks:
    verdict = (
        f"On a stratified subsample of {n} Probe-1 cells (seed={SUBSAMPLE_SEED}, "
        f"Qwen2.5-1.5B-Instruct), fp16 vs 4-bit NF4 showed mean |Δ mean_logprob|={mad_lp}, "
        f"Spearman(mean_logprob)={sp}, median |Δ gold_first_token_rank|={med}, "
        f"and 95th-percentile rank shift={p95}. Because the median absolute rank shift "
        f"exceeds ~{RANK_DROP_THRESHOLD} vocabulary positions, 4-bit rank measurements are "
        f"NOT usable for paper claims: drop all 4-bit gold_first_token_rank results and "
        f"retain fp16 (and, where needed, 8-bit) ranks only; state this explicitly in Methods. "
        f"Pairwise table: O6_quantization_sensitivity.csv."
    )
else:
    verdict = (
        f"On a stratified subsample of {n} Probe-1 cells (seed={SUBSAMPLE_SEED}, "
        f"Qwen2.5-1.5B-Instruct), fp16 vs 4-bit NF4 showed mean |Δ mean_logprob|={mad_lp}, "
        f"Spearman(mean_logprob)={sp}, median |Δ gold_first_token_rank|={med}, "
        f"and 95th-percentile rank shift={p95}. The median absolute rank shift is ≤ "
        f"~{RANK_DROP_THRESHOLD} positions, so 4-bit rank measurements are usable as a "
        f"robustness check alongside fp16 primary results, with the tabulated sensitivity "
        f"bounds reported in Methods. Pairwise table: O6_quantization_sensitivity.csv."
    )

O6_SUMMARY_TXT.write_text(verdict + "\n")
print("\n=== SUMMARY (also written to O6_quantization_sensitivity_summary.txt) ===")
print(verdict)
print(f"\n[decision] drop_4bit_ranks={drop_4bit_ranks}  threshold={RANK_DROP_THRESHOLD}")


## Download / Drive backup

Copy into the repo after Colab:
- `O6_quantization_sensitivity.csv` → `results/derived/O6_quantization_sensitivity.csv`
- `O6_quantization_sensitivity_items.csv` → `results/raw/O6_quantization_sensitivity_items.csv`
- `O6_quantization_sensitivity_summary.txt` → `results/derived/O6_quantization_sensitivity_summary.txt`


In [ ]:
_out_files = [
    O6_CSV,
    O6_ITEMS_CSV,
    O6_SUMMARY_TXT,
    OUT_DIR / "O6_subsample_manifest.json",
]
_drive_dir = Path("/content/drive/MyDrive/rvc_colab_out")
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)
try:
    import shutil as _shutil
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for p in _out_files:
        if p.exists():
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
except Exception as exc:
    print("[backup] skipped:", exc)
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in _out_files:
        if p.exists():
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
except Exception as exc:
    print("[download] skipped (not Colab or download blocked):", exc)
